# Characterize a dataset + its FDs (Table 1 / §3.1)

Profiles a dataset and the FDs declared over it, then drops FDs whose LHS is too
sparse to support a repair pattern. Reports the §3.1 / Table 1 style measures
(G3 error per FD, LHS redundancy, value lengths, attribute overlap) and lets you
inspect the LHS distribution or violation clusters of any single FD.

Two variants, selected by the **`LAZY`** toggle:

- **`LAZY = True`** — streaming path for large parquet/CSV tables via
  `eval.report.characterize_lazy` (reads only `attr(fd)` per FD; never loads the
  whole table). This variant can also write the filtered FD set back to disk.
- **`LAZY = False`** — eager path for small CSV tables via
  `eval.report.characterize` (falls back to `characterize_dataset` +
  `characterize_fds` when the FDs and table come from different folders), with a
  richer per-FD LHS-distribution view.

**Inputs → Outputs:** `datasets/<name>/{fds.csv, clean.csv|clean.parquet}` →
printed characterization + LHS-redundancy filter (kept / dropped FDs) + one
per-FD inspection view. Optionally (OFF by default) overwrites the source
`fds.csv` with the filtered FD set.

**Config knobs:**
- `LAZY` — streaming/parquet path (True) vs. eager/CSV path (False).
- `MIN_REDUNDANCY` — drop FDs whose LHS redundancy is below this.
- `fds_path`, `data_path` — the dataset files to profile.
- `WRITE_BACK` — opt-in to overwrite `fds.csv` with the filtered set.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

# horizon/ pipeline modules use flat imports (`from fds.fd import ...`) and need
# horizon/ on sys.path; eval + package imports (`horizon.fds.fd`) need the repo
# root. Put both on the path, repo root first (else `import horizon` binds to
# horizon/horizon.py, not the package).
REPO = Path("..").resolve()
HZN = REPO / "horizon"
for p in (str(HZN), str(REPO)):
    if p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(HZN))
sys.path.insert(0, str(REPO))

import polars as pl

from horizon.utils.loaders import get_fds, load_table
from eval.dataset_eval import characterize_dataset
from eval.fd_eval import characterize_fds, fd_lhs_redundancy
from eval.report import characterize, characterize_lazy

## Configuration

In [ ]:
# --- config ---
LAZY = False            # True: streaming/parquet (characterize_lazy); False: eager/CSV (characterize)
MIN_REDUNDANCY = 0.1    # drop FDs whose LHS redundancy is below this (~n^MIN_REDUNDANCY rows/group)

# dataset files to profile — edit these two paths
if LAZY:
    fds_path = Path("..") / "datasets" / "complaints_10m" / "fds.csv"
    data_path = Path("..") / "datasets" / "complaints_10m" / "clean.parquet"
else:
    fds_path = Path("..") / "datasets" / "hospital_2k" / "fds.csv"
    data_path = Path("..") / "datasets" / "hospital_2k" / "clean.csv"

## Load the FDs and the data

In [ ]:
fds = get_fds(fds_path, data_path)
print(f"Loaded {len(fds)} FDs")

if LAZY:
    # never materialize the whole table; just show its schema
    scan = pl.scan_parquet if data_path.suffix.lower() == ".parquet" else pl.scan_csv
    print(scan(data_path).collect_schema().names())
    df = None
else:
    df = load_table(data_path)
    print(df.head())

fds

## Characterize + LHS redundancy

In [ ]:
# LAZY: characterize_lazy reads only attr(fd) per FD, prints progress, and returns
#       fd_lhs_redundancy among its measures.
# eager: compute LHS redundancy directly from the in-memory df.
if LAZY:
    result = characterize_lazy(data_path, fds)
    red = result["fd_lhs_redundancy"]
else:
    result = None
    red = fd_lhs_redundancy(df, fds)

## Drop FDs below `MIN_REDUNDANCY`

In [ ]:
# drop FDs whose LHS redundancy is below MIN_REDUNDANCY — too few repeats to
# support a pattern (avg LHS group size ≈ n^MIN_REDUNDANCY).
kept = [fd for fd in fds if red[fd.lhs_attributes] >= MIN_REDUNDANCY]
dropped = [fd for fd in fds if red[fd.lhs_attributes] < MIN_REDUNDANCY]
print(f"kept {len(kept)} / {len(fds)} FDs; dropping {len(dropped)} (red < {MIN_REDUNDANCY}):")
for fd in dropped:
    print(f"  {red[fd.lhs_attributes]:.3f}  {fd}")

# LAZY: prune the characterization dicts to the kept FDs (eager re-characterizes below)
if LAZY:
    result["g3_error"] = {fd: v for fd, v in result["g3_error"].items() if fd in kept}
    result["violation_clusters"] = {fd: v for fd, v in result["violation_clusters"].items() if fd in kept}
    result["fd_lhs_redundancy"] = {lhs: v for lhs, v in red.items() if any(fd.lhs_attributes == lhs for fd in kept)}

fds = kept

## Optional: write the filtered FDs back to `fds.csv`

**OFF by default.** Set `WRITE_BACK = True` to overwrite the source `fds.csv`
with the kept FD set — a destructive edit to dataset files, so it never runs
unless you opt in.

In [ ]:
WRITE_BACK = False  # opt-in: overwrite the source fds.csv with the filtered FDs

if WRITE_BACK:
    import pandas as pd

    pd.DataFrame(
        [{"from": ";".join(fd.lhs_attributes), "to": fd.rhs} for fd in fds]
    ).to_csv(fds_path, index=False)
    print(f"wrote {len(fds)} FDs -> {fds_path}")
else:
    print("WRITE_BACK is off; source fds.csv left unchanged")

## Dataset + FD summary

In [ ]:
# LAZY: summarize the lazy characterization (omit the bulky violation_clusters).
# eager: FD-side metrics only make sense when the FDs were defined for *this* table;
#        proxy for "they belong together" is that both files live in the same folder.
if LAZY:
    summary = {k: v for k, v in result.items() if k not in {"violation_clusters"}}
    pprint(summary, sort_dicts=False)
elif fds_path.parent == data_path.parent:
    pprint(characterize(df, fds), sort_dicts=False)
else:
    print(
        f"skipped combined FDs: {fds_path.parent.name}/fds.csv and "
        f"{data_path.parent.name}/{data_path.name} are from different datasets"
    )
    pprint(characterize_dataset(df), sort_dicts=False)
    pprint(characterize_fds(fds), sort_dicts=False)

## Inspect one FD

In [ ]:
# inspect one FD by index
i = 13 if LAZY else 7
fd = fds[i]

if LAZY:
    # show the violation clusters computed by characterize_lazy
    clusters = result["violation_clusters"][fd]
    print(f"[{i}] {fd}   G3 = {result['g3_error'][fd]:.4f}   {clusters.height} violation rows")
    pl.Config.set_fmt_str_lengths(200)  # don't truncate string cells in the display
    pl.Config.set_tbl_rows(300)         # show all rows, not Polars' default 10
    display(clusters.tail(300))
else:
    # each distinct LHS tuple once with its occurrence count, sorted by the LHS
    # columns so duplicates / outliers are easy to scan.
    def lhs_view(fd, *, n=30):
        cols = list(fd.lhs_attributes)  # always a tuple of column names
        return df.group_by(cols).len(name="count").sort(cols).head(n)

    print(f"[{i}] {fd}   lhs_redundancy = {red[fd.lhs_attributes]:.3f}")
    display(lhs_view(fd, n=30))